# 04 · Dashboard, fault test và tổng kết

Notebook cuối đọc các kết quả đã tạo, dựng dashboard Plotly và chạy một bài **fault test mức dữ liệu**: xoá một partition Parquet, quan sát kho mất dữ liệu, rồi khôi phục và kiểm tra lại.

Đây là mô phỏng ở mức file/partition, không phải kill node HDFS thật. Nó chứng minh cùng một nguyên tắc: hệ thống phải phát hiện thiếu dữ liệu và phục hồi được.


In [1]:
import sys, json, shutil
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from config import *
ensure_dirs()
setup_matplotlib()

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown, HTML

from warehouse import connect


## 1. Đọc lại kết quả từ các notebook trước

Notebook này không huấn luyện lại; nó là lớp trình bày và kiểm tra. Nếu file thiếu, chạy lại notebook 00–03 trước.


In [2]:
missing = []
for name in [
    "benchmark_results.csv", "yield_metrics.json", "yield_predictions.csv",
    "yield_feature_importance.csv", "anomaly_counts.csv", "anomaly_top.csv",
    "forecast_metrics.csv",
]:
    if not (REPORTS_DIR / name).exists():
        missing.append(name)
print("Thiếu file:", missing if missing else "không")

bench = pd.read_csv(REPORTS_DIR / "benchmark_results.csv")
yield_metrics = json.loads((REPORTS_DIR / "yield_metrics.json").read_text(encoding="utf-8"))
pred = pd.read_csv(REPORTS_DIR / "yield_predictions.csv")
importance = pd.read_csv(REPORTS_DIR / "yield_feature_importance.csv")
anomaly_counts = pd.read_csv(REPORTS_DIR / "anomaly_counts.csv")
anomaly_top = pd.read_csv(REPORTS_DIR / "anomaly_top.csv")
forecast_metrics = pd.read_csv(REPORTS_DIR / "forecast_metrics.csv")

con = connect()
yield_country = con.execute("""
    SELECT country_iso3, year, cereal_yield_kg_per_ha
    FROM fact_cereal_annual ORDER BY country_iso3, year
""").df()
print("benchmark rows:", len(bench), "| yield metrics:", len(yield_metrics["metrics"]))


Thiếu file: không
benchmark rows: 8 | yield metrics: 9


## 2. Dashboard Plotly

Bốn góc nhìn: năng suất theo quốc gia, thời gian benchmark, số anomaly theo địa điểm, và so sánh forecast với persistence. File HTML được lưu để mở ngoài notebook.


In [3]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Năng suất ngũ cốc theo quốc gia",
        "Thời gian 4 truy vấn benchmark",
        "Số anomaly theo địa điểm",
        "Forecast 7 ngày: Ridge vs persistence (MAE)",
    ),
    vertical_spacing=0.16, horizontal_spacing=0.10,
)

palette = [PALETTE["green"], PALETTE["teal"], PALETTE["gold"], PALETTE["danger"],
           PALETTE["muted"], PALETTE["green_light"]]
for i, (country, group) in enumerate(yield_country.groupby("country_iso3")):
    fig.add_trace(go.Scatter(
        x=group["year"], y=group["cereal_yield_kg_per_ha"], mode="lines+markers",
        name=country, line=dict(width=1.6, color=palette[i % len(palette)]),
        showlegend=False,
    ), row=1, col=1)

for scale, group in bench.groupby("scale"):
    fig.add_trace(go.Bar(
        x=group["query"], y=group["seconds_median"], name=f"{scale:,} dòng",
        marker_color=PALETTE["green"] if scale == bench["scale"].max() else PALETTE["green_light"],
    ), row=1, col=2)

fig.add_trace(go.Bar(
    x=anomaly_counts["location_id"], y=anomaly_counts["anomalies"],
    name="anomaly", marker_color=PALETTE["teal"], showlegend=False,
), row=2, col=1)

for model, color in (("Ridge AR", PALETTE["gold"]), ("Persistence", PALETTE["muted"])):
    part = forecast_metrics[forecast_metrics["model"] == model]
    fig.add_trace(go.Bar(
        x=part["location_id"], y=part["MAE"], name=model, marker_color=color,
    ), row=2, col=2)

fig.update_layout(
    height=820, template="plotly_white", barmode="group",
    margin=dict(l=40, r=30, t=70, b=50),
    title_text="Agri DWH · dashboard mô phỏng",
    legend=dict(orientation="h", y=-0.10),
)
fig.update_yaxes(title_text="kg/ha", row=1, col=1)
fig.update_yaxes(title_text="giây", row=1, col=2)
fig.update_yaxes(title_text="số giờ", row=2, col=1)
fig.update_yaxes(title_text="MAE", row=2, col=2)

dashboard_path = REPORTS_DIR / "dashboard.html"
fig.write_html(dashboard_path, include_plotlyjs=True)
print("dashboard:", dashboard_path)
fig.show()


dashboard: D:\projectDW\agri-dwh\reports\dashboard.html


## 3. Fault test: mất một partition và khôi phục

Xoá tạm thư mục `year=2020/month=6` khỏi lake Parquet. Một connection mới sẽ không thấy dữ liệu tháng đó. Sau đó khôi phục thư mục và kiểm tra số dòng trở lại đúng.

Đây là bài kiểm tra **data-level recovery**: nó không chứng minh HDFS replication, nhưng chứng minh pipeline phát hiện và phục hồi được mất mát ở tầng partition.


In [4]:
part = PARQUET_DIR / "fact_weather_hourly" / "year=2020" / "month=6"
backup = WAREHOUSE_DIR / "_fault_backup_2020_06"
if backup.exists():
    if not part.exists():
        shutil.move(str(backup), str(part))
    else:
        shutil.rmtree(backup)

def count_month():
    c = duckdb.connect()
    try:
        pattern = (part / "*.parquet").as_posix()
        return c.execute(
            f"SELECT COUNT(*) FROM read_parquet('{pattern}')"
        ).fetchall()[0][0]
    except Exception:
        return 0
    finally:
        c.close()

before = count_month()
shutil.move(str(part), str(backup))
after_delete = count_month()
shutil.move(str(backup), str(part))
after_restore = count_month()

fault = pd.DataFrame([
    {"stage": "before", "rows_2020_06": before, "status": "ok"},
    {"stage": "partition removed", "rows_2020_06": after_delete, "status": "degraded" if after_delete < before else "unexpected"},
    {"stage": "partition restored", "rows_2020_06": after_restore, "status": "recovered" if after_restore == before else "failed"},
])
fault.to_csv(REPORTS_DIR / "fault_test.csv", index=False)
display(fault)


,stage,rows_2020_06,status
0,before,8640,ok
1,partition removed,0,degraded
2,partition restored,8640,recovered


## 4. Tổng kết số liệu

Tổng hợp metric chính từ notebook 01–03 thành một bản tóm tắt để nộp hoặc đưa vào báo cáo.


In [5]:
best = yield_metrics["best_model"]
test_row = next(
    m for m in yield_metrics["metrics"]
    if m["model"] == best + " (refit)" and "test" in m["split"]
)
baseline = next(
    m for m in yield_metrics["metrics"]
    if m["model"] == "Baseline mean" and "test" in m["split"]
)
fc = forecast_metrics.pivot(index="location_id", columns="model", values="MAE")
fc["ridge_better"] = fc["Ridge AR"] < fc["Persistence"]

lines = [
    "# Tổng kết Agri DWH",
    "",
    "## Dữ liệu",
    "- 1.052.064 dòng hourly, 12 địa điểm, 2015–2024.",
    "- 120 partition year-month; không null/duplicate ở khoá chính.",
    "",
    "## Benchmark",
    f"- 4 truy vấn trên {bench['scale'].min():,} và {bench['scale'].max():,} dòng.",
    "- Thời gian và kết quả lưu tại `reports/benchmark_results.csv`.",
    "",
    "## Yield model",
    f"- Model: **{best}**.",
    f"- Test MAE: **{test_row['MAE']:.1f} kg/ha**, R²: **{test_row['R2']:.3f}**, MAPE: **{test_row['MAPE_pct']:.1f}%**.",
    f"- Baseline mean MAE: **{baseline['MAE']:.1f} kg/ha**.",
    "- Cảnh báo: 24 mẫu test; longitude là feature quan trọng nhất, nên model học khác biệt vùng/quốc gia rất mạnh. Không diễn giải là quan hệ nhân quả khí hậu–năng suất.",
    "",
    "## Anomaly",
    f"- Tổng anomaly: **{int(anomaly_counts['anomalies'].sum()):,}** giờ trên 1.052.064 dòng.",
    "- Sự kiện Cần Thơ 14/06/2020 được đánh dấu; inject anomaly test có recall 1.0.",
    "",
    "## Forecast",
    "| Địa điểm | Ridge MAE | Persistence MAE | Ridge tốt hơn? |",
    "|---|---:|---:|---|",
]
for loc, row in fc.iterrows():
    lines.append(
        f"| {loc} | {row['Ridge AR']:.4f} | {row['Persistence']:.4f} | "
        f"{'có' if row['ridge_better'] else 'không'} |"
    )
lines += [
    "",
    "Persistence là baseline rất mạnh cho độ ẩm đất 7 ngày; Ridge chỉ thắng ở FR-CVL. Kết quả này được giữ trung thực, không chỉnh test để đẹp.",
    "",
    "## Fault test",
    f"- Trước khi xoá partition: {before:,} dòng tháng 6/2020.",
    f"- Sau khi xoá: {after_delete:,}; sau khôi phục: {after_restore:,}.",
    "- Đây là fault test mức partition/file, không phải kill DataNode.",
    "",
    "## Khác biệt với bài báo",
    "- DuckDB + Parquet thay cho Hive/Spark/HDFS; file-based stream thay cho Kafka; Plotly thay cho Superset/Grafana; notebook runner thay cho Airflow.",
    "- Dữ liệu là thật (NASA POWER + World Bank), không phải dữ liệu tổng hợp.",
]
summary_md = "\n".join(lines)
summary_path = REPORTS_DIR / "final_summary.md"
summary_path.write_text(summary_md, encoding="utf-8")
display(Markdown(summary_md))
print("summary:", summary_path)


# Tổng kết Agri DWH

## Dữ liệu
- 1.052.064 dòng hourly, 12 địa điểm, 2015–2024.
- 120 partition year-month; không null/duplicate ở khoá chính.

## Benchmark
- 4 truy vấn trên 100,000 và 1,052,064 dòng.
- Thời gian và kết quả lưu tại `reports/benchmark_results.csv`.

## Yield model
- Model: **GradientBoosting**.
- Test MAE: **359.8 kg/ha**, R²: **0.969**, MAPE: **9.6%**.
- Baseline mean MAE: **2000.0 kg/ha**.
- Cảnh báo: 24 mẫu test; longitude là feature quan trọng nhất, nên model học khác biệt vùng/quốc gia rất mạnh. Không diễn giải là quan hệ nhân quả khí hậu–năng suất.

## Anomaly
- Tổng anomaly: **5,440** giờ trên 1.052.064 dòng.
- Sự kiện Cần Thơ 14/06/2020 được đánh dấu; inject anomaly test có recall 1.0.

## Forecast
| Địa điểm | Ridge MAE | Persistence MAE | Ridge tốt hơn? |
|---|---:|---:|---|
| FR-CVL | 0.0098 | 0.0106 | có |
| US-IA | 0.0164 | 0.0136 | không |
| VN-CT | 0.0108 | 0.0100 | không |

Persistence là baseline rất mạnh cho độ ẩm đất 7 ngày; Ridge chỉ thắng ở FR-CVL. Kết quả này được giữ trung thực, không chỉnh test để đẹp.

## Fault test
- Trước khi xoá partition: 8,640 dòng tháng 6/2020.
- Sau khi xoá: 0; sau khôi phục: 8,640.
- Đây là fault test mức partition/file, không phải kill DataNode.

## Khác biệt với bài báo
- DuckDB + Parquet thay cho Hive/Spark/HDFS; file-based stream thay cho Kafka; Plotly thay cho Superset/Grafana; notebook runner thay cho Airflow.
- Dữ liệu là thật (NASA POWER + World Bank), không phải dữ liệu tổng hợp.

summary: D:\projectDW\agri-dwh\reports\final_summary.md


## Kết luận notebook 04

- Dashboard HTML đã được lưu; mở `reports/dashboard.html` bằng trình duyệt.
- Fault test cho thấy partition bị mất được phát hiện và khôi phục đúng số dòng.
- Bản tóm tắt cuối nằm ở `reports/final_summary.md`.
- Đây là bản **lite** chạy được trên Windows; xem `README.md` để biết cách nâng cấp lên Kafka/Spark/Hive nếu có Docker.


In [6]:
con.close()
